# LemmaSlop on Colab: math RAG with Ternary Bonsai 27B, speculative decoding and narrated Manim videos

This notebook runs the [LemmaSlop](https://github.com/DS-PNQ/mandatoryslopragapp) app end to end inside a Colab GPU runtime. Nothing leaves the runtime except the one-time downloads.

1. **Ternary Bonsai 2.5 27B** answers every question. It runs on PrismML's llama.cpp build with the model's own **multi-token-prediction (MTP) draft head**, so decoding is speculative: the head guesses up to 3 tokens and the 27B model checks them in one pass. PrismML measures 1.5–1.8× faster decoding on NVIDIA GPUs with output that matches plain decoding. If that build cannot start, the notebook falls back to stock llama.cpp with the `Q2_0` file.
2. **Retrieval** ranks passages with a dense embedder (**Qwen3-Embedding-0.6B**, or **EmbeddingGemma 2** when GPU memory is tight), with the app's LSA + e-means + BM25 index as tie-breaker and fallback, then reorders the candidates with the **Qwen3-Reranker-0.6B** cross-encoder when it fits. Uploads are tokenized by the app's Rust core.
3. **Manim** (with LaTeX) renders an animation for any question that asks for one, and **MioTTS-0.6B** narrates it. The model writes the scene in a separate step; a scene that fails is sent back for a fix twice, and if it still fails the app renders a storyboard of the answer, so every animation request ends in a video.
4. The **Next.js app** is built and started on port 3000. Open it through Colab's private port proxy, a public Cloudflare quick tunnel, or ask from the notebook (section 9).

**Before you run it:** *Runtime → Change runtime type → T4 GPU* (or L4/A100), then *Runtime → Run all*.

**Setup time** on a free T4: about 15 minutes, most of it the 6.2 GB model download, the LaTeX install and the voice model. The app builds in the background meanwhile. Optional: an `HF_TOKEN` Colab secret (key icon in the left sidebar, notebook access on) speeds up Hugging Face downloads.

## 1. Settings

In [ ]:
# --- Source ------------------------------------------------------------------
REPO_URL    = "https://github.com/DS-PNQ/mandatoryslopragapp.git"
REPO_BRANCH = ""                     # "" = the repository's default branch

# --- Stock llama.cpp (embeddings, reranker, MioTTS, LFM3, and the Bonsai fallback)
#   "conda"  - prebuilt CUDA build from conda-forge, about 2 minutes, nothing compiled (default)
#   "source" - compile llama.cpp for this GPU, 20-25 minutes on the free tier's 2 CPUs
LLAMA_INSTALL   = "conda"
LLAMA_MIN_BUILD = 11960              # b11960: LFM3 support (and newer than the b11556 Bonsai's Q2_0 needs)
LLAMA_CPP_REF   = "latest"           # source builds only
CACHE_TO_DRIVE  = False              # keep a source-built llama-server in Google Drive between sessions
DRIVE_CACHE     = "/content/drive/MyDrive/lemmaslop-cache"

# --- Main model: Ternary Bonsai 2.5 27B ----------------------------------------
# prism-ml/Ternary-Bonsai-2.5-27B-GGUF (Apache 2.0): a ternary-weight Qwen3.8-27B
# reasoning model with vision. Its "-mtp" files carry a multi-token-prediction
# draft head for speculative decoding (the head is inert when drafting is off).
INSTALL_BONSAI = True
BONSAI_REPO    = "prism-ml/Ternary-Bonsai-2.5-27B-GGUF"
#   "auto"  - PrismML's llama.cpp build with PTQ1_0 + MTP; falls back to "stock" if it fails
#   "prism" - only the PrismML build
#   "stock" - stock llama.cpp with the Q2_0 file (no MTP head support)
BONSAI_RUNTIME = "auto"
PRISM_TAG      = "prism-b12178-03ec600"    # the release PrismML's Bonsai-demo pins
PRISM_CUDA     = "12.4"                    # its CUDA 12.4 x64 build has sm_70/sm_80 code (T4, L4, A100) and needs glibc 2.34
BONSAI_PRISM_GGUF = "Ternary-Bonsai-2.5-27B-PTQ1_0-mtp.gguf"   # 6.2 GB, the fastest packing on NVIDIA GPUs
BONSAI_STOCK_GGUF = "Ternary-Bonsai-2.5-27B-Q2_0.gguf"         # 7.7 GB, for stock llama.cpp
BONSAI_MMPROJ  = "Ternary-Bonsai-2.5-27B-mmproj-Q8_0.gguf"     # "" to skip image input (saves ~0.9 GB of VRAM)
BONSAI_CTX     = 40960                     # one slot: room for its reasoning plus the passages
BONSAI_PORT    = 8095

# Speculative decoding:
#   "mtp"       - the model's MTP draft head (PrismML build). The CUDA preset: 3 drafts, min probability 0.1
#   "mtp+ngram" - also draft from n-grams already in the context (helps when the model repeats text,
#                 e.g. a scene repair that re-emits most of the code). Experimental.
#   "off"       - plain decoding
BONSAI_SPEC      = "mtp"
BONSAI_SPEC_NMAX = 3
BONSAI_SPEC_PMIN = 0.1

# --- Retrieval models -----------------------------------------------------------
#   A dense embedder (fused with BM25 + LSA by RRF) and Qwen3-Reranker-0.6B
#   (cross-encoder over the fused candidates), all Apache 2.0.
#   "auto" starts each on the GPU when there is room left after Bonsai; the embedder
#   falls back to the CPU, the reranker is skipped. True forces both on the GPU.
INSTALL_RETRIEVAL = "auto"
#   "auto"             - Qwen3-Embedding-0.6B when it fits on the GPU (it retrieved best:
#                        92% top-1 on the app's 52-question evaluation), else
#                        EmbeddingGemma 2 (77%, but half the size and 2.5x faster on a CPU)
#   "qwen3"            - Qwen3-Embedding-0.6B: 640 MB, 1024-d
#   "embeddinggemma-2" - EmbeddingGemma 2 (Oct 2026): 310 MB, 768-d; needs a llama.cpp
#                        that knows its architecture, else the other model is used
EMBEDDER = "auto"
EMBEDDERS = {
    "embeddinggemma-2": ("ggml-org/embeddinggemma-2-GGUF", "embeddinggemma-2-Q8_0.gguf", "mean", 900),
    "qwen3": ("Qwen/Qwen3-Embedding-0.6B-GGUF", "Qwen3-Embedding-0.6B-Q8_0.gguf", "last", 1400),
}
RERANK_REPO, RERANK_GGUF = "ggml-org/Qwen3-Reranker-0.6B-Q8_0-GGUF", "qwen3-reranker-0.6b-q8_0.gguf"
EMBED_PORT, RERANK_PORT  = 8097, 8098

# --- Optional second model: LFM3 1.7B ----------------------------------------
INSTALL_LFM3 = False                 # True also serves LFM3 1.7B and lists it in the model picker
GGUF_REPO    = "LiquidAI/LFM3-1.7B-GGUF"
GGUF_QUANTS  = ["Q8_0", "F16", "BF16", "Q6_K", "Q5_K_M", "Q4_K_M"]
LLAMA_PORT   = 8091                  # not 8080: Colab's own Jupyter server listens there

# --- Animations ----------------------------------------------------------------
INSTALL_MANIM = True                 # Manim Community, about 2 minutes
MANIM_LATEX   = True                 # TeX Live for MathTex, about 3 minutes; without it formulas are drawn as text
MANIM_QUALITY = "m"                  # l = 480p, m = 720p, h = 1080p
INSTALL_TTS   = True                 # MioTTS-0.6B narration voice
TTS_GGUF    = "MioTTS-0.6B-Q8_0.gguf"          # from Aratako/MioTTS-GGUF
TTS_CODEC   = "Aratako/MioCodec-25Hz-24kHz"    # the codec the 0.6B model card names
# The voice is cloned from the model card's English reference clip. That clip was
# itself synthesized with gemini-2.5-pro-tts, so check its terms before commercial
# use, or set TTS_REFERENCE_WAV to a clip of your own (WAV/FLAC/OGG, under 20 s).
TTS_REFERENCE_URL = "https://huggingface.co/Aratako/MioTTS-0.6B/resolve/main/samples/en_ref1.wav"
TTS_REFERENCE_WAV = ""

# --- Access --------------------------------------------------------------------
APP_PORT    = 3000
OPEN_TUNNEL = True                   # also print a public trycloudflare.com link

from pathlib import Path
WORK = Path("/content/lemmaslop")    # a Path, so re-running this cell alone keeps later cells working

## 2. Helpers and environment check

In [ ]:
import glob, json, os, re, shutil, subprocess, sys, time, urllib.request
from pathlib import Path

WORK = Path(WORK); WORK.mkdir(parents=True, exist_ok=True)
LOGS = WORK / "logs"; LOGS.mkdir(exist_ok=True)
MODELS = WORK / "models"; MODELS.mkdir(exist_ok=True)

def sh(cmd, cwd=None, env=None, quiet=False):
    """Run a shell command, stream its output, raise if it fails."""
    print(f"$ {cmd}", flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, env=env, text=True,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    tail = []
    for line in p.stdout:
        tail = (tail + [line])[-40:]
        if not quiet:
            print(line, end="", flush=True)
    if p.wait() != 0:
        if quiet:
            print("".join(tail))
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}")

def secret(name):
    """Colab secret (if notebook access is on), then environment variable, else None."""
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            return v
    except Exception:
        pass
    return os.environ.get(name) or None

def free_port(port, tries=50):
    """First port from `port` up that nothing on this runtime is listening on."""
    import socket
    for p in range(port, port + tries):
        with socket.socket() as sock:
            sock.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
            try:
                sock.bind(("127.0.0.1", p))
                return p
            except OSError:
                continue
    raise RuntimeError(f"no free port in {port}-{port + tries - 1}")

def wait_http(url, timeout, proc=None, log=None, ok=lambda r: r.status == 200):
    t0 = time.time()
    while time.time() - t0 < timeout:
        if proc is not None and proc.poll() is not None:
            print(Path(log).read_text()[-4000:] if log else "")
            raise RuntimeError(f"process exited with code {proc.returncode} before {url} came up")
        try:
            with urllib.request.urlopen(url, timeout=5) as r:
                if ok(r):
                    return round(time.time() - t0)
        except Exception:
            pass
        time.sleep(2)
    raise TimeoutError(f"{url} not ready after {timeout}s")

def post_json(url, body, timeout=600):
    req = urllib.request.Request(url, data=json.dumps(body).encode(), headers={"content-type": "application/json"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.load(r)

def gpu_free_mib():
    out = subprocess.run("nvidia-smi --query-gpu=memory.free --format=csv,noheader,nounits",
                         shell=True, capture_output=True, text=True).stdout.split()
    return int(out[0]) if out else 0

def hf_file(repo, name):
    from huggingface_hub import hf_hub_download
    return hf_hub_download(repo, name, token=secret("HF_TOKEN"), local_dir=str(MODELS))

smi = subprocess.run("nvidia-smi --query-gpu=name,compute_cap,memory.total,driver_version --format=csv,noheader",
                     shell=True, capture_output=True, text=True).stdout.strip().splitlines()
assert smi, "No GPU. Runtime -> Change runtime type -> pick a GPU, then run again."
GPU_NAME, COMPUTE_CAP, GPU_MEM, DRIVER = [x.strip() for x in smi[0].split(",")]
CUDA_ARCH = COMPUTE_CAP.replace(".", "")                 # "7.5" -> "75"
GPU_GIB = int(re.search(r"\d+", GPU_MEM).group(0)) / 1024
print(f"GPU: {GPU_NAME} (sm_{CUDA_ARCH}, {GPU_MEM}), driver {DRIVER}, {os.cpu_count()} CPUs")

NVCC = shutil.which("nvcc") or ("/usr/local/cuda/bin/nvcc" if Path("/usr/local/cuda/bin/nvcc").exists() else None)
CUDA_VERSION = (re.search(r"release ([\d.]+)", subprocess.run(f"{NVCC} --version", shell=True, capture_output=True,
                text=True).stdout) or [None, "unknown"])[1] if NVCC else "none"
print("nvcc:", NVCC, "| CUDA", CUDA_VERSION)

## 3. Node.js 22 and the app build

Next.js 16 needs Node 20.9 or newer, and the Node that ships with Colab is older. This installs Node 22 LTS into `/content/lemmaslop/node`, clones the app, and starts `npm ci` and `npm run build` in the background, so the app builds while the models download.

In [ ]:
NODE_DIR = WORK / "node"
if not (NODE_DIR / "bin" / "node").exists():
    releases = json.load(urllib.request.urlopen("https://nodejs.org/dist/index.json"))
    v = next(r["version"] for r in releases if r["version"].startswith("v22."))
    tarball = f"node-{v}-linux-x64.tar.xz"
    sh(f"curl -fsSL https://nodejs.org/dist/{v}/{tarball} -o {WORK}/{tarball}")
    sh(f"mkdir -p {NODE_DIR} && tar -xJf {WORK}/{tarball} -C {NODE_DIR} --strip-components=1 && rm {WORK}/{tarball}")
os.environ["PATH"] = f"{NODE_DIR}/bin:" + os.environ["PATH"]
sh("node --version && npm --version")

APP = WORK / "app"
if not (APP / "package.json").exists():
    branch = f"--branch {REPO_BRANCH}" if REPO_BRANCH else ""
    sh(f"git clone --depth 1 {branch} {REPO_URL} {APP}")
sh("git log --oneline -1", cwd=APP)

# Every answer is generated on this runtime: drop hosted-model keys so the app
# never offers Opus or routes LFM3 through OpenRouter.
HOSTED_KEYS = {"ANTHROPIC_API_KEY", "ANTHROPIC_AUTH_TOKEN", "OPENROUTER_API_KEY", "LFM3_API_KEY"}
BASE_ENV = {k: v for k, v in os.environ.items() if k not in HOSTED_KEYS}
BASE_ENV["NEXT_TELEMETRY_DISABLED"] = "1"

# devDependencies (tsx, typescript) are needed to build the index, so install
# in development mode and build in production mode.
APP_BUILD_LOG = LOGS / "app-build.log"
app_build = subprocess.Popen(
    "NODE_ENV=development npm ci --no-audit --no-fund && NODE_ENV=production npm run build",
    shell=True, cwd=APP, env=BASE_ENV, stdout=open(APP_BUILD_LOG, "w"), stderr=subprocess.STDOUT)
print("App build started in the background; log:", APP_BUILD_LOG)

## 4. Stock llama.cpp with CUDA

With `LLAMA_INSTALL = "conda"`, llama.cpp comes from conda-forge's CUDA 12 build, installed with micromamba into `/content/lemmaslop/llama-env`. It has native code for sm_75 (T4), sm_80 (A100), sm_89 (L4) and sm_90 (H100) and needs only glibc 2.17. It serves the embedding and reranking models, MioTTS, LFM3 if you turn it on, and Bonsai if PrismML's build cannot start. The conda environment also provides the CUDA 12 runtime libraries that PrismML's build links against (section 6).

If that install fails, or with `LLAMA_INSTALL = "source"`, llama.cpp is compiled as a single `llama-server` for this GPU's architecture only, and cached in Drive with `CACHE_TO_DRIVE = True`.

In [ ]:
def install_conda():
    """Prebuilt CUDA llama.cpp from conda-forge; returns the llama-server path."""
    env_dir = WORK / "llama-env"
    server = env_dir / "bin" / "llama-server"
    if server.exists():
        return str(server)
    mamba = WORK / "micromamba"
    if not mamba.exists():
        sh(f"curl -fsSL https://conda.anaconda.org/conda-forge/linux-64/micromamba-2.10.0-0.tar.bz2 "
           f"| tar -xj -C {WORK} bin/micromamba && mv {WORK}/bin/micromamba {mamba} && rmdir {WORK}/bin")
    driver_cuda = re.search(r"CUDA Version:\s*([\d.]+)", subprocess.run("nvidia-smi", shell=True,
                            capture_output=True, text=True).stdout)
    menv = {**os.environ, "MAMBA_ROOT_PREFIX": str(WORK / "mamba-root")}
    if driver_cuda:
        menv["CONDA_OVERRIDE_CUDA"] = driver_cuda.group(1)   # the solver's view of the driver
    spec = f"llama.cpp[version='>={LLAMA_MIN_BUILD}',build='cuda12*']"
    t0 = time.time()
    sh(f'{mamba} create -y -q -p {env_dir} -c conda-forge --override-channels "{spec}"', env=menv, quiet=True)
    print(f"installed in {(time.time() - t0) / 60:.1f} min")
    return str(server)


def build_from_source():
    """Compile llama-server for this GPU only, reusing a Drive-cached build when one fits."""
    ref = LLAMA_CPP_REF
    cache_dir = None
    if CACHE_TO_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        cache_dir = Path(DRIVE_CACHE); cache_dir.mkdir(parents=True, exist_ok=True)

    def cache_path(tag):
        return cache_dir / f"llama-server-{tag}-sm{CUDA_ARCH}-cuda{CUDA_VERSION}" if cache_dir else None

    if ref == "latest" and cache_dir is not None:
        pattern = re.compile(rf"llama-server-b(\d+)-sm{CUDA_ARCH}-cuda{re.escape(CUDA_VERSION)}$")
        cached = sorted((int(m.group(1)) for p in cache_dir.iterdir() if (m := pattern.match(p.name))), reverse=True)
        if cached and cached[0] >= LLAMA_MIN_BUILD:
            ref = f"b{cached[0]}"
    if ref == "latest":
        out = subprocess.run("git ls-remote --tags --refs https://github.com/ggml-org/llama.cpp 'b*'",
                             shell=True, capture_output=True, text=True).stdout
        tags = re.findall(r"refs/tags/(b\d+)$", out, re.M)
        ref = max(tags, key=lambda t: int(t[1:])) if tags else "master"
    print("llama.cpp", ref)

    server = str(WORK / "llama-server")
    cached = cache_path(ref)
    if cached is not None and cached.exists() and ref != "master":
        print("Using cached build from Drive:", cached)
        shutil.copy(cached, server)                     # Drive does not keep the exec bit
        os.chmod(server, 0o755)
        return server
    if Path(server).exists():
        return server
    assert NVCC, "CUDA toolkit (nvcc) not found; use a GPU runtime."
    src = WORK / "llama.cpp"
    if not src.exists():
        sh(f"git clone --depth 1 --branch {ref} https://github.com/ggml-org/llama.cpp {src}")
    build_env = {**os.environ, "PATH": f"{Path(NVCC).parent}:" + os.environ["PATH"], "CUDACXX": NVCC}
    cuda_home = Path(NVCC).resolve().parent.parent
    candidates = [*glob.glob(f"{cuda_home}/lib64/stubs/libcuda.so"),
                  *glob.glob(f"{cuda_home}/targets/*/lib/stubs/libcuda.so"),
                  *glob.glob("/usr/local/cuda*/lib64/stubs/libcuda.so"),
                  *glob.glob("/usr/local/cuda*/targets/*/lib/stubs/libcuda.so")]
    candidates += re.findall(r"=>\s*(\S*libcuda\.so(?:\.1)?)$",
                             subprocess.run("ldconfig -p", shell=True, capture_output=True, text=True).stdout, re.M)
    candidates += glob.glob("/usr/lib64-nvidia/libcuda.so*") + glob.glob("/usr/lib/x86_64-linux-gnu/libcuda.so*")
    libcuda = next((c for c in candidates if os.path.isfile(c)), None)
    driver_flag = f"-DCUDA_cuda_driver_LIBRARY={libcuda}" if libcuda else "-DGGML_CUDA_NO_VMM=ON"
    shutil.rmtree(src / "build", ignore_errors=True)
    generator = "-G Ninja" if shutil.which("ninja") else ""
    sh(f"cmake -B build {generator} -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES={CUDA_ARCH} -DBUILD_SHARED_LIBS=OFF "
       f"{driver_flag} -DLLAMA_CURL=OFF -DLLAMA_BUILD_TESTS=OFF -DCMAKE_BUILD_TYPE=Release",
       cwd=src, env=build_env, quiet=True)
    t0 = time.time()
    print(f"Compiling on {os.cpu_count()} CPUs; this is the slow step.", flush=True)
    sh(f"cmake --build build --config Release --target llama-server -j{os.cpu_count()}", cwd=src, env=build_env, quiet=True)
    print(f"built {ref} in {(time.time() - t0) / 60:.1f} min")
    shutil.copy(src / "build" / "bin" / "llama-server", server)
    if cached is not None and ref != "master":
        shutil.copy(server, cached)
    return server


LLAMA_SERVER = None
if LLAMA_INSTALL == "conda":
    try:
        LLAMA_SERVER = install_conda()
    except Exception as e:
        print(f"Prebuilt install failed ({e}); compiling from source instead.")
if LLAMA_SERVER is None:
    LLAMA_SERVER = build_from_source()
sh(f"{LLAMA_SERVER} --version")
LLAMA_HELP = subprocess.run([LLAMA_SERVER, "--help"], capture_output=True, text=True).stdout

## 5. Manim, LaTeX and the MioTTS voice

**Manim** goes into `/content/lemmaslop/manim-venv`, so Colab's own packages are left alone. With `MANIM_LATEX = True` a minimal TeX Live is installed too, so formulas in videos are typeset (`MathTex`); the app's runner draws a formula as text if TeX cannot compile it, so one bad formula no longer fails a video. The app checks each scene before running it and renders it in a restricted interpreter with CPU, memory and time limits.

**MioTTS-0.6B** reads the narration: the model is served by llama-server on the GPU, behind the [MioTTS-Inference](https://github.com/Aratako/MioTTS-Inference) API with the MioCodec-25Hz-24kHz codec. This takes 3 to 5 minutes the first time.

In [ ]:
MANIM_PY = None
if INSTALL_MANIM:
    venv = WORK / "manim-venv"
    py = venv / "bin" / "python"
    if not py.exists() or subprocess.run([str(py), "-c", "import manim"], capture_output=True).returncode != 0:
        t0 = time.time()
        sh("apt-get -qq update && apt-get -qq install -y libpango1.0-dev pkg-config ffmpeg > /dev/null", quiet=True)
        if subprocess.run([sys.executable, "-m", "venv", str(venv)]).returncode != 0:
            py = Path(sys.executable)      # no venv module: install into Colab's Python instead
        sh(f"{py} -m pip install -q manim", quiet=True)
        print(f"Manim installed in {time.time() - t0:.0f}s")
    if MANIM_LATEX and not (shutil.which("latex") and shutil.which("dvisvgm")):
        t0 = time.time()
        sh("apt-get -qq install -y --no-install-recommends texlive-latex-base texlive-latex-recommended "
           "texlive-latex-extra texlive-fonts-recommended dvisvgm > /dev/null", quiet=True)
        print(f"TeX Live installed in {time.time() - t0:.0f}s")
    version = subprocess.run([str(py), "-c", "import manim; print(manim.__version__)"],
                             capture_output=True, text=True).stdout.strip()
    tex_dir = WORK / "tex-check"; tex_dir.mkdir(exist_ok=True)
    tex_ok = subprocess.run([str(py), "-c", "from manim import MathTex; MathTex(r'\\sum_{n=0}^{\\infty} r^n = \\frac{1}{1-r}')"],
                            cwd=tex_dir, capture_output=True, text=True).returncode == 0
    print(f"Manim {version} ({py}); MathTex {'works' if tex_ok else 'unavailable, formulas are drawn as text'}")
    MANIM_PY = str(py)
else:
    print("Skipping Manim: animation answers will show the scene code instead of a video.")

In [ ]:
MIOTTS_URL = None
if INSTALL_TTS:
    t0 = time.time()
    from huggingface_hub import hf_hub_download
    tts_gguf = hf_hub_download("Aratako/MioTTS-GGUF", TTS_GGUF, token=secret("HF_TOKEN"))

    # 1. The MioTTS LLM on llama-server (the binary from step 4).
    subprocess.run(["pkill", "-f", "llama-server.*--alias MioTTS"])
    TTS_LLM_PORT = free_port(8093)
    TTS_LLM_LOG = LOGS / "miotts-llm.log"
    tts_llm = subprocess.Popen(
        [LLAMA_SERVER, "-m", tts_gguf, "--alias", "MioTTS", "--host", "127.0.0.1", "--port", str(TTS_LLM_PORT),
         "-ngl", "999", "-c", "4096", "--cont-batching"],
        stdout=open(TTS_LLM_LOG, "w"), stderr=subprocess.STDOUT, start_new_session=True)
    wait_http(f"http://127.0.0.1:{TTS_LLM_PORT}/health", 300, tts_llm, TTS_LLM_LOG)

    # 2. The MioTTS-Inference API (codec + presets) in its own uv environment.
    TTS_REPO = WORK / "MioTTS-Inference"
    if not TTS_REPO.exists():
        sh(f"git clone -q --depth 1 https://github.com/Aratako/MioTTS-Inference.git {TTS_REPO}")
    sh("pip install -q uv", quiet=True)
    sh("uv sync -q", cwd=TTS_REPO, quiet=True)

    # 3. An English voice preset encoded with the 24 kHz codec.
    presets = TTS_REPO / "presets-24k"
    if not (presets / "en_female.pt").exists():
        ref = Path(TTS_REFERENCE_WAV) if TTS_REFERENCE_WAV else WORK / "miotts-en-ref.wav"
        if not TTS_REFERENCE_WAV:
            urllib.request.urlretrieve(TTS_REFERENCE_URL, ref)
        sh(f"uv run python scripts/generate_preset.py --audio {ref} --preset-id en_female "
           f"--model-id {TTS_CODEC} --output-dir {presets} --device cuda", cwd=TTS_REPO, quiet=True)

    # 4. Start the API.
    subprocess.run(["pkill", "-f", "run_server.py --llm-base-url"])
    TTS_PORT = free_port(8094)
    TTS_LOG = LOGS / "miotts.log"
    tts_proc = subprocess.Popen(
        f"uv run python run_server.py --llm-base-url http://127.0.0.1:{TTS_LLM_PORT}/v1 --llm-model MioTTS "
        f"--codec-model {TTS_CODEC} --presets-dir {presets} --device cuda --host 127.0.0.1 --port {TTS_PORT}",
        shell=True, cwd=TTS_REPO, stdout=open(TTS_LOG, "w"), stderr=subprocess.STDOUT, start_new_session=True)
    wait_http(f"http://127.0.0.1:{TTS_PORT}/health", 600, tts_proc, TTS_LOG)
    MIOTTS_URL = f"http://127.0.0.1:{TTS_PORT}"

    req = urllib.request.Request(f"{MIOTTS_URL}/v1/tts", headers={"content-type": "application/json"},
        data=json.dumps({"text": "The geometric series converges when the ratio is less than one.",
                         "reference": {"type": "preset", "preset_id": "en_female"},
                         "output": {"format": "wav"}}).encode())
    sample = WORK / "miotts-sample.wav"
    sample.write_bytes(urllib.request.urlopen(req, timeout=300).read())
    print(f"MioTTS ready at {MIOTTS_URL} in {time.time() - t0:.0f}s")
    from IPython.display import Audio, display
    display(Audio(str(sample)))
else:
    print("Skipping MioTTS: animations get captions but no voice.")

## 6. Ternary Bonsai 2.5 27B with speculative decoding

**Why MTP.** Speculative decoding needs a cheap guesser whose guesses the big model accepts often. For Bonsai 2.5 the best guesser is the one PrismML trained into the model: the `-mtp.gguf` files carry a multi-token-prediction head (one extra layer) that drafts the next few tokens from the model's own hidden state, and the 27B model verifies them in a single batched pass, keeping only tokens it would have produced itself. There is no second model to download or fit in VRAM (the head adds 0.2 GB), and PrismML reports mean KL divergence between drafting on and off of 5×10⁻⁵, i.e. the same output. Their CUDA measurements, 3 drafts at minimum probability 0.1: 1.5–1.8× faster decode (A100 105 → 171 tok/s, RTX 4090 111 → 204 tok/s), with prefill 9–23% slower because the head makes its own pass over the prompt. LemmaSlop's answers are long (Bonsai reasons for hundreds of tokens before it writes), so decode dominates and MTP pays.

**Alternatives considered.** A separate small draft model would need Bonsai's tokenizer and costs VRAM the T4 does not have; PrismML has not released an official drafter for 2.5. Prompt n-gram drafting (`ngram-mod`) costs nothing but only drafts when the output repeats the context, so it is offered as an add-on (`BONSAI_SPEC = "mtp+ngram"`), not a replacement.

**How it runs here.** MTP drafting needs PrismML's llama.cpp build, so this section downloads its pinned release (`PRISM_TAG`), a CUDA 12.4 build whose sm_70/sm_80 code runs on the T4, L4 and A100 and whose glibc 2.34 requirement Colab's Ubuntu 22.04 meets. It links CUDA 12's runtime libraries, taken from the conda environment of section 4 (or pip's `nvidia-*-cu12` wheels). Each step is checked with a real request: the smoke test must answer correctly, and the response's `timings` must show drafts being made. If anything fails, the notebook retries without drafting, then falls back to stock llama.cpp with the `Q2_0` file, so Bonsai always comes up.

In [ ]:
def cuda12_lib_dirs():
    """Directories that hold libcudart.so.12 and libcublas(Lt).so.12."""
    need = ["libcudart.so.12", "libcublas.so.12", "libcublasLt.so.12"]
    def scan():
        dirs = [str(WORK / "llama-env" / "lib"), *glob.glob(str(WORK / "cu12" / "nvidia" / "*" / "lib")),
                *glob.glob("/usr/local/lib/python3*/dist-packages/nvidia/*/lib")]
        found = {}
        for d in dirs:
            for n in need:
                if n not in found and Path(d, n).exists():
                    found[n] = d
        return found
    found = scan()
    if len(found) < len(need):
        sh(f"{sys.executable} -m pip install -q --target {WORK / 'cu12'} nvidia-cuda-runtime-cu12 nvidia-cublas-cu12", quiet=True)
        found = scan()
    missing = [n for n in need if n not in found]
    if missing:
        raise RuntimeError(f"CUDA 12 libraries not found: {missing}")
    return sorted(set(found.values()))


def install_prism():
    """PrismML's llama.cpp release (MTP drafting, PTQ1_0 kernels); returns (llama-server, env)."""
    d = WORK / "prism" / PRISM_TAG
    server = d / "llama-server"
    if not server.exists():
        asset = f"llama-{PRISM_TAG}-bin-linux-cuda-{PRISM_CUDA}-x64.tar.gz"
        d.mkdir(parents=True, exist_ok=True)
        sh(f"curl -fsSL https://github.com/PrismML-Eng/llama.cpp/releases/download/{PRISM_TAG}/{asset} "
           f"| tar -xz -C {d} --strip-components=1", quiet=True)
    env = {**os.environ, "LD_LIBRARY_PATH": ":".join([str(d), *cuda12_lib_dirs(), os.environ.get("LD_LIBRARY_PATH", "")])}
    missing = subprocess.run(f"ldd {server} | grep 'not found' || true", shell=True, env=env,
                             capture_output=True, text=True).stdout.strip()
    if missing:
        raise RuntimeError(f"PrismML llama-server is missing libraries:\n{missing}")
    help_text = subprocess.run([str(server), "--help"], env=env, capture_output=True, text=True).stdout
    if "draft-mtp" not in help_text:
        print("Warning: this PrismML build does not list draft-mtp")
    return str(server), env


def spec_args(runtime, help_text):
    """llama-server flags for the chosen speculative decoding mode."""
    if BONSAI_SPEC == "off":
        return []
    types = []
    if runtime == "prism" and "mtp" in BONSAI_SPEC:
        types.append("draft-mtp")
    if ("ngram" in BONSAI_SPEC or runtime == "stock") and "ngram-mod" in help_text:
        types.append("ngram-mod")
    if not types:
        return []
    args = ["--spec-type", ",".join(types)]
    if "draft-mtp" in types:
        args += ["--spec-draft-n-min", "0", "--spec-draft-n-max", str(BONSAI_SPEC_NMAX),
                 "--spec-draft-p-min", str(BONSAI_SPEC_PMIN)]
    if "ngram-mod" in types:
        # Shorter n-gram drafts than the MoE-oriented defaults, as the docs advise for dense models.
        args += ["--spec-ngram-mod-n-match", "24", "--spec-ngram-mod-n-min", "16", "--spec-ngram-mod-n-max", "32"]
    return args


PRISM = None
want_bonsai = INSTALL_BONSAI is True or (INSTALL_BONSAI == "auto" and GPU_GIB >= 14)
if want_bonsai and BONSAI_RUNTIME in ("auto", "prism"):
    try:
        PRISM = install_prism()
        print("PrismML llama.cpp", PRISM_TAG, "ready")
    except Exception as e:
        if BONSAI_RUNTIME == "prism":
            raise
        print(f"PrismML build unavailable ({e}); Bonsai will run on stock llama.cpp with Q2_0.")

In [ ]:
BONSAI_URL, BONSAI_INFO = None, {}
BONSAI_ALIAS = "Ternary-Bonsai-2.5-27B"

def start_bonsai(server, env, model, extra, label):
    """Starts llama-server for Bonsai and smoke-tests it; returns the base URL or raises."""
    global BONSAI_PORT
    subprocess.run(["pkill", "-f", f"llama-server.*--alias {BONSAI_ALIAS}"])
    time.sleep(2)
    BONSAI_PORT = free_port(BONSAI_PORT)
    log = LOGS / "bonsai.log"
    mmproj = ["--mmproj", hf_file(BONSAI_REPO, BONSAI_MMPROJ)] if BONSAI_MMPROJ else []
    # -np 1: one request at a time, so the server does not start four slots that
    # share (and enlarge) the KV cache. Sampling is the model card's thinking preset.
    cmd = [server, "-m", model, *mmproj, "--alias", BONSAI_ALIAS, "--jinja", "--host", "127.0.0.1",
           "--port", str(BONSAI_PORT), "-ngl", "99", "-fa", "on", "-c", str(BONSAI_CTX), "-np", "1",
           "--temp", "1.0", "--top-p", "0.95", "--top-k", "20", "--min-p", "0", *extra]
    print(f"Starting Bonsai ({label}):", " ".join(cmd[1:]).replace(str(MODELS) + "/", ""))
    proc = subprocess.Popen(cmd, env=env, stdout=open(log, "w"), stderr=subprocess.STDOUT, start_new_session=True)
    secs = wait_http(f"http://127.0.0.1:{BONSAI_PORT}/health", 900, proc, log)
    url = f"http://127.0.0.1:{BONSAI_PORT}/v1"
    r = post_json(f"{url}/chat/completions", {
        "model": BONSAI_ALIAS, "max_tokens": 4096,
        "messages": [{"role": "user", "content": "What is the derivative of x^3? One line."}]})
    answer = (r["choices"][0]["message"].get("content") or "").strip()
    t = r.get("timings", {})
    flat = re.sub(r"[\s${}]|\\cdot", "", answer)
    if not any(f in flat for f in ("3x^2", "3x²", "3x**2")):
        raise RuntimeError(f"smoke test answer looks wrong: {answer[:200]!r}")
    drafted, accepted = t.get("draft_n") or 0, t.get("draft_n_accepted") or 0
    info = {"label": label, "ready_s": secs, "tok_s": round(t.get("predicted_per_second") or 0, 1),
            "drafted": drafted, "accepted": accepted,
            "acceptance": round(accepted / drafted, 2) if drafted else None}
    print(f"Bonsai ready in {secs}s. Answer: {answer[:80]!r}")
    print(f"decode {info['tok_s']} tok/s" + (f", drafts accepted {accepted}/{drafted} ({info['acceptance']:.0%})"
                                             if drafted else ", no drafting"))
    if any("draft-mtp" in a for a in extra) and not drafted:
        print("Warning: MTP drafting is configured but this response reports no drafts (timings.draft_n).")
    return url, info

if want_bonsai:
    attempts = []
    if PRISM:
        prism_model = None
        try:
            prism_model = hf_file(BONSAI_REPO, BONSAI_PRISM_GGUF)
        except Exception as e:
            print(f"Could not download {BONSAI_PRISM_GGUF}: {e}")
        if prism_model:
            server, env = PRISM
            args = spec_args("prism", subprocess.run([server, "--help"], env=env, capture_output=True, text=True).stdout)
            if args:
                attempts.append((server, env, prism_model, args, f"PrismML {PRISM_TAG}, PTQ1_0, {' '.join(args[:2])}"))
            attempts.append((server, env, prism_model, [], f"PrismML {PRISM_TAG}, PTQ1_0, no drafting"))
    if BONSAI_RUNTIME != "prism":
        attempts.append((LLAMA_SERVER, dict(os.environ), None, spec_args("stock", LLAMA_HELP), "stock llama.cpp, Q2_0"))
    for server, env, model, extra, label in attempts:
        try:
            if model is None:
                model = hf_file(BONSAI_REPO, BONSAI_STOCK_GGUF)
            BONSAI_URL, BONSAI_INFO = start_bonsai(server, env, model, extra, label)
            break
        except Exception as e:
            print(f"--> {label} failed: {e}\n{(LOGS / 'bonsai.log').read_text()[-1500:]}\n")
            if extra and model is not None and label.startswith("stock"):
                try:   # stock build without n-gram drafting
                    BONSAI_URL, BONSAI_INFO = start_bonsai(server, env, model, [], "stock llama.cpp, Q2_0, no drafting")
                    break
                except Exception as e2:
                    print(f"--> stock without drafting failed: {e2}")
    if not BONSAI_URL:
        raise RuntimeError("Bonsai did not start; see the messages above and logs/bonsai.log")
    sh("nvidia-smi --query-gpu=memory.used,memory.total --format=csv")
else:
    print(f"Skipping Ternary Bonsai ({GPU_NAME}). Set INSTALL_BONSAI = True to run it.")

### 6b. Optional: LFM3 1.7B

In [ ]:
LFM3_URL = None
if INSTALL_LFM3:
    from huggingface_hub import list_repo_files
    files = [f for f in list_repo_files(GGUF_REPO, token=secret("HF_TOKEN")) if f.lower().endswith(".gguf")]
    QUANT, GGUF_FILE = next((q, f) for q in GGUF_QUANTS for f in files
                            if re.search(rf"[-_.]{re.escape(q)}\.gguf$", f, re.I) and "mmproj" not in f.lower())
    MODEL_PATH = hf_file(GGUF_REPO, GGUF_FILE)
    MODEL_NAME = f"LFM3-1.7B-{QUANT}"
    subprocess.run(["pkill", "-f", "llama-server.*--alias LFM3"])
    LLAMA_PORT = free_port(LLAMA_PORT)
    LLAMA_LOG = LOGS / "llama-server.log"
    llama_proc = subprocess.Popen(
        [LLAMA_SERVER, "-m", MODEL_PATH, "--alias", MODEL_NAME, "--jinja", "--host", "127.0.0.1",
         "--port", str(LLAMA_PORT), "-ngl", "999", "-c", "40960", "-np", "1"],
        stdout=open(LLAMA_LOG, "w"), stderr=subprocess.STDOUT, start_new_session=True)
    print(f"LFM3 ready in {wait_http(f'http://127.0.0.1:{LLAMA_PORT}/health', 600, llama_proc, LLAMA_LOG)}s")
    LFM3_URL = f"http://127.0.0.1:{LLAMA_PORT}/v1"
else:
    print("LFM3 is off (INSTALL_LFM3 = False).")

## 7. Retrieval models: embeddings and reranker

The app's own index (LSA + e-means IVF + BM25) needs no server. These two small models improve on it, following what recent retrieval work finds strongest for its cost: **hybrid** lexical + dense retrieval fused with reciprocal rank fusion, then a **cross-encoder rerank** of the fused candidates.

- **An embedder**, chosen by `EMBEDDER = "auto"`: Qwen3-Embedding-0.6B when it fits on the GPU, otherwise EmbeddingGemma 2, which is half the size and faster on a CPU. On the app's 52-question evaluation Qwen3 put the right passage first 92% of the time and EmbeddingGemma 2 77%; both had it in the top six 98–100% of the time. Each gets its own prompt format (EmbeddingGemma 2: `task: search result | query: …` and `title: … | text: …`; Qwen3: an instruction on queries). The dense order decides the ranking; BM25 and LSA only break ties and add candidates. Passages carry a "note § section" or "file, page" header so a chunk is embedded with its context, and LaTeX is verbalised (`\sum` → "sum"). Uploads are embedded once and cached. The comparison behind the default is in the README ("Choosing the embedder").
- **Qwen3-Reranker-0.6B** reads the question and each of the 20 fused candidates together and keeps the best 6.

Each server has a timeout in the app and is skipped for a minute if it fails, so a crash here never blocks an answer.

In [ ]:
EMBED_URL = RERANK_URL = None

def start_aux(name, repo, gguf, port, flags, gpu):
    subprocess.run(["pkill", "-f", f"llama-server.*--alias {name}"])
    port = free_port(port)
    log = LOGS / f"{name}.log"
    proc = subprocess.Popen([LLAMA_SERVER, "-m", hf_file(repo, gguf), "--alias", name, "--host", "127.0.0.1",
                             "--port", str(port), "-ngl", "999" if gpu else "0", *flags],
                            stdout=open(log, "w"), stderr=subprocess.STDOUT, start_new_session=True)
    wait_http(f"http://127.0.0.1:{port}/health", 300, proc, log)
    return f"http://127.0.0.1:{port}/v1"

if INSTALL_RETRIEVAL:
    free = gpu_free_mib()
    force = INSTALL_RETRIEVAL is True
    print(f"{free} MiB of GPU memory free")
    if EMBEDDER == "auto":
        order = ["qwen3", "embeddinggemma-2"] if force or free >= EMBEDDERS["qwen3"][3] else ["embeddinggemma-2", "qwen3"]
    else:
        order = [EMBEDDER] + [k for k in EMBEDDERS if k != EMBEDDER]
    rerank_gpu = force or free >= EMBEDDERS[order[0]][3] + 1400
    EMBED_MODEL = None
    for name in order:
        repo, gguf, pooling, need_mib = EMBEDDERS[name]
        alias = "embeddinggemma-2" if name == "embeddinggemma-2" else "qwen3-embedding-0.6b"
        embed_gpu = force or gpu_free_mib() >= need_mib
        try:
            EMBED_URL = start_aux(alias, repo, gguf, EMBED_PORT,
                                  ["--embedding", "--pooling", pooling, "-c", "8192", "-b", "8192", "-ub", "8192"], embed_gpu)
            v = post_json(f"{EMBED_URL}/embeddings", {"input": ["task: search result | query: ratio test"]})["data"][0]["embedding"]
            EMBED_MODEL = alias
            print(f"{alias} ready on the {'GPU' if embed_gpu else 'CPU'} ({len(v)} dimensions)")
            break
        except Exception as e:
            EMBED_URL = None
            print(f"{alias} failed ({e}); {'trying the next embedder' if name != order[-1] else 'the app keeps LSA + BM25'}")
    if rerank_gpu:
        try:
            RERANK_URL = start_aux("qwen3-reranker-0.6b", RERANK_REPO, RERANK_GGUF, RERANK_PORT,
                                   ["--reranking", "-c", "8192", "-b", "4096", "-ub", "4096"], True)
            r = post_json(f"{RERANK_URL}/rerank", {"query": "What is the ratio test?",
                          "documents": ["The ratio test compares |a_{n+1}/a_n| with 1.", "Paris is in France."]})
            best = max(r["results"], key=lambda x: x["relevance_score"])["index"]
            print("Reranker ready" + ("" if best == 0 else " (warning: it ranked the wrong document first)"))
        except Exception as e:
            RERANK_URL = None
            print("Reranker failed, the app keeps the fused ranking:", e)
    else:
        print("Not enough GPU memory left for the reranker; skipped.")
    sh("nvidia-smi --query-gpu=memory.used,memory.total --format=csv")

## 8. Start the app

In [ ]:
t0 = time.time()
if app_build.wait() != 0:
    print(APP_BUILD_LOG.read_text()[-4000:])
    raise RuntimeError("npm ci / npm run build failed; see the log above")
print(f"App build finished (waited {time.time() - t0:.0f}s here)")

APP_ENV = {**BASE_ENV,
           "NODE_ENV": "production",
           **({"BONSAI_BASE_URL": BONSAI_URL, "BONSAI_MODEL": BONSAI_ALIAS,
               "BONSAI_VISION": "1" if BONSAI_MMPROJ else "0"} if BONSAI_URL else {}),
           **({"LFM3_BASE_URL": LFM3_URL, "LFM3_MODEL": MODEL_NAME} if LFM3_URL else {}),
           **({"LEMMA_EMBED_URL": EMBED_URL, "LEMMA_EMBED_MODEL": EMBED_MODEL} if EMBED_URL else {}),
           **({"LEMMA_RERANK_URL": RERANK_URL} if RERANK_URL else {}),
           **({"MANIM_PYTHON": MANIM_PY, "MANIM_QUALITY": MANIM_QUALITY} if MANIM_PY else {}),
           **({"MIOTTS_URL": MIOTTS_URL, "MIOTTS_PRESET": "en_female"} if MIOTTS_URL else {})}

APP_LOG = LOGS / "app.log"
subprocess.run(["pkill", "-f", f"next start -p {APP_PORT}"])
time.sleep(1)
APP_PORT = free_port(APP_PORT)
app_proc = subprocess.Popen(f"npx next start -p {APP_PORT} -H 0.0.0.0", shell=True, cwd=APP, env=APP_ENV,
                            stdout=open(APP_LOG, "w"), stderr=subprocess.STDOUT, start_new_session=True)
secs = wait_http(f"http://127.0.0.1:{APP_PORT}/", 180, app_proc, APP_LOG)
print(f"LemmaSlop is up on port {APP_PORT} ({secs}s)")

## 9. Ask from the notebook

`ask()` calls the app's `/api/ask` endpoint (the same one the web page uses), prints its status lines as they stream, renders the answer with its sources, and plays the video when the question asks for an animation.

In [ ]:
from IPython.display import HTML, Markdown, display
import base64

def ask(question, engine="bonsai", history=None, show_sources=True, animate=False):
    body = {"question": question, "engine": engine, "history": history or [], "animate": animate}
    req = urllib.request.Request(f"http://127.0.0.1:{APP_PORT}/api/ask", data=json.dumps(body).encode(),
                                 headers={"content-type": "application/json"})
    sources, trace, transcript, last, anims, t0 = [], None, "", None, {}, time.time()
    with urllib.request.urlopen(req, timeout=1800) as r:
        for raw in r:
            if not raw.strip():
                continue
            ev = json.loads(raw)
            if ev["type"] == "a2ui":
                upd = ev["message"].get("updateDataModel")
                if not upd:
                    continue
                path, value = upd["path"], upd["value"]
                if path == "/status" and value["text"] != last:
                    last = value["text"]
                    print(f"[{time.time() - t0:6.1f}s] {last}", flush=True)
                elif path == "/sources":
                    sources = value
                elif path == "/trace":
                    trace = value
                elif path.startswith("/anims/"):
                    note = value.get("note") or value["status"]
                    if anims.get(path, {}).get("_note") != note:
                        print(f"[{time.time() - t0:6.1f}s] animation: {note}", flush=True)
                    anims[path] = {**value, "_note": note}
            elif ev["type"] == "error":
                raise RuntimeError(ev["message"])
            elif ev["type"] == "done":
                transcript = ev["transcript"]
    if trace:
        neural = " + ".join(x for x in [trace.get("embedder"), trace.get("reranker")] if x)
        print(f"model {trace['model']} | scanned {trace['scanned']}/{trace['total']} passages "
              f"in {len(trace['cells'])}/{trace['k']} e-means cells | neural: {neural or 'off'}")
    display(Markdown(transcript or "_(empty answer)_"))
    for a in anims.values():
        if a["status"] == "ready":
            mp4 = urllib.request.urlopen(f"http://127.0.0.1:{APP_PORT}{a['src']}").read()
            tag = "storyboard of the answer" if a.get("fallback") else "model-written scene"
            display(HTML(f'<p><b>Animation</b> ({tag}, {"narrated" if a.get("voiced") else "captions only"})</p>'
                         f'<video controls width="720" src="data:video/mp4;base64,{base64.b64encode(mp4).decode()}"></video>'))
        else:
            print(f"Animation {a['status']}: {a.get('message', '')}")
    if show_sources and sources:
        display(Markdown("\n".join(f"- **[{s['n']}]** {s['note']} §{s['section']} {s['heading']}" for s in sources)))
    return {"question": question, "answer": transcript}

turn = ask("Why does the geometric series converge exactly when |r| < 1?")

In [ ]:
# A follow-up that uses the previous turn as context.
ask("How does the ratio test generalise that?", history=[turn])

In [ ]:
# An animation: the answer streams first, then the scene is written, voiced and rendered.
ask("Animate how the partial sums of the geometric series approach 1/(1-r)")

## 10. Open the web UI

**Colab proxy (private):** the first link goes through Colab's port proxy, so it only works in a browser signed in to the same Google account. If it shows a blank page or a cookie error, allow third-party cookies for `colab.research.google.com` and `*.googleusercontent.com`, or use the tunnel link.

**Cloudflare quick tunnel (public):** the second link works from any browser, but anyone who has it can use the app while the runtime is up. Stop it with the last cell.

In [ ]:
from google.colab.output import eval_js
PROXY_URL = eval_js(f"google.colab.kernel.proxyPort({APP_PORT}, {{'cache': false}})")
links = [f"**Colab proxy (only you):** [{PROXY_URL}]({PROXY_URL})"]

if OPEN_TUNNEL:
    CF = WORK / "cloudflared"
    if not CF.exists():
        sh(f"curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o {CF} && chmod +x {CF}")
    TUNNEL_LOG = LOGS / "cloudflared.log"
    subprocess.run(["pkill", "-f", "cloudflared tunnel"])
    tunnel_proc = subprocess.Popen([str(CF), "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{APP_PORT}"],
                                   stdout=open(TUNNEL_LOG, "w"), stderr=subprocess.STDOUT, start_new_session=True)
    PUBLIC_URL = None
    for _ in range(60):
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", TUNNEL_LOG.read_text())
        if m:
            PUBLIC_URL = m.group(0); break
        time.sleep(1)
    if PUBLIC_URL:
        try:
            wait_http(PUBLIC_URL, 60)      # the hostname takes a few seconds to resolve
        except Exception:
            pass
        links.append(f"**Cloudflare tunnel (anyone with the link):** [{PUBLIC_URL}]({PUBLIC_URL})")
    else:
        print("Tunnel did not start:\n", TUNNEL_LOG.read_text()[-2000:])

spec = BONSAI_INFO.get("label", "")
display(Markdown("### LemmaSlop is live\n\n" + "\n\n".join(links) +
                 f"\n\nAnswers come from Ternary Bonsai 2.5 27B on this GPU ({spec})"
                 + (", or LFM3 1.7B (pick it next to the send button)." if LFM3_URL else ".")))

## 11. Logs and shutdown

In [ ]:
# Tail the logs if something looks off.
for log in sorted(LOGS.glob("*.log")):
    print(f"===== {log.name} =====")
    print("".join(log.read_text(errors="replace").splitlines(keepends=True)[-25:]))

In [ ]:
# Stop everything (Runtime -> Disconnect and delete runtime also works).
for pattern in ["cloudflared tunnel", f"next start -p {APP_PORT}", "llama-server", "run_server.py --llm-base-url"]:
    subprocess.run(["pkill", "-f", pattern])
print("stopped")